# 3.3 Document Loaders & Text Splitters

## Playground Notebook

Every RAG pipeline starts here: **loading** raw data from diverse sources into standardized `Document` objects, then **splitting** that text into optimally-sized chunks for embeddings and retrieval.

| Topic | What You'll Learn |
|-------|-------------------|
| **Document Object** | The universal `page_content` + `metadata` format |
| **File Loaders** | PDF, Word, CSV, TXT, JSON loaders |
| **Web Loaders** | Loading from URLs and web pages |
| **Directory Loader** | Batch loading entire folders |
| **Custom Loaders** | Building your own loader with `BaseLoader` |
| **RecursiveCharacterTextSplitter** | The recommended default splitter |
| **CharacterTextSplitter** | Single-separator splitting |
| **TokenTextSplitter** | Token-count-based splitting |
| **Language-Specific Splitters** | Code-aware splitting |
| **MarkdownHeaderTextSplitter** | Structure-aware splitting |
| **Two-Pass Splitting** | Structure-first, then size enforcement |
| **Chunk Size Optimization** | Finding the sweet spot for your use case |

### The Pipeline

```
Raw Data (PDF, Web, DB, ...)
        |
  Document Loader  -->  List[Document(page_content, metadata)]
        |
    Text Splitter   -->  List[Document] (smaller, chunked)
        |
  Embedding + Vector Store  -->  RAG-ready!
```

> **Model:** `llama3.2:3b` via **Ollama** — running 100% locally, no API keys needed.

---

In [1]:
# ============================================================
#  INSTALL DEPENDENCIES (run once)
# ============================================================
# !pip install langchain langchain-core langchain-ollama langchain-text-splitters
# !pip install python-docx pypdf beautifulsoup4 tiktoken

In [2]:
import os
import json
import tempfile
from pathlib import Path
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from IPython.display import display, Markdown

# ============================================================
#  CONFIGURATION
# ============================================================
MODEL = "llama3.2:3b"

llm = ChatOllama(model=MODEL, temperature=0.7)

# Quick connectivity test
test = llm.invoke("Say 'ready' in one word.")
print(f"\u2705 Connected to Ollama | Model: {MODEL}")
print(f"   Response: {test.content}")

✅ Connected to Ollama | Model: llama3.2:3b
   Response: Ready.


---

## 1. The Document Object — Universal Data Format

Every document loader in LangChain produces `Document` objects with two fields:

| Field | Type | Description |
|-------|------|-------------|
| `page_content` | `str` | The extracted text content |
| `metadata` | `dict` | Info about the source (filename, page, URL, etc.) |

This standardization means **all downstream components** (splitters, embeddings, retrievers) work the same regardless of the original source.

### Why Metadata Matters

- **Source attribution** — Tell users *which document and page* answered their question
- **Filtering** — Search only documents from a specific department or date range
- **Access control** — Enforce permissions based on document classification
- **Debugging** — Trace retrieval failures back to the source document

### Experiment 1: Creating and Inspecting Document Objects

In [3]:
# Document objects are simple — just text + metadata

doc1 = Document(
    page_content="LangChain is a framework for building LLM-powered applications.",
    metadata={"source": "langchain_docs.pdf", "page": 1, "author": "Harrison Chase"}
)

doc2 = Document(
    page_content="Document loaders extract text from diverse file formats.",
    metadata={"source": "tutorial.html", "section": "loaders", "last_updated": "2024-01-15"}
)

# Inspect
for doc in [doc1, doc2]:
    print(f"Content:  {doc.page_content}")
    print(f"Metadata: {doc.metadata}")
    print(f"Type:     {type(doc).__name__}")
    print()

Content:  LangChain is a framework for building LLM-powered applications.
Metadata: {'source': 'langchain_docs.pdf', 'page': 1, 'author': 'Harrison Chase'}
Type:     Document

Content:  Document loaders extract text from diverse file formats.
Metadata: {'source': 'tutorial.html', 'section': 'loaders', 'last_updated': '2024-01-15'}
Type:     Document



---

## 2. File-Based Document Loaders

### PDF Loaders — Choosing the Right One

| Loader | Library | Strengths | Weaknesses |
|--------|---------|-----------|------------|
| `PyPDFLoader` | PyPDF2 | Fast, lightweight, page-by-page | Struggles with tables, multi-column |
| `PyMuPDFLoader` | PyMuPDF | Fast, rich metadata | Requires system dependencies |
| `PDFPlumberLoader` | pdfplumber | Excellent table extraction | Slower for large documents |
| `UnstructuredPDFLoader` | Unstructured | Handles complex layouts | Heavier dependency |

**How to choose:**
- Simple text PDFs (reports, articles) -> `PyPDFLoader`
- Table-heavy documents (financial, invoices) -> `PDFPlumberLoader`
- Complex layouts (research papers) -> `UnstructuredPDFLoader`

### Experiment 2A: Loading a Text File

In [4]:
from langchain_community.document_loaders import TextLoader

# Create a sample text file to load
sample_text = """Introduction to Document Loaders

Document loaders are the entry point of any RAG pipeline. They extract
text and metadata from diverse sources — PDFs, web pages, databases,
and more — into a standardized Document format.

The key insight is that regardless of the source format, the output
is always the same: a list of Document objects with page_content
and metadata fields.

This standardization allows all downstream components to work
identically whether the data came from a PDF or a website."""

# Write sample file
sample_path = os.path.join(tempfile.gettempdir(), "sample_doc.txt")
with open(sample_path, "w", encoding="utf-8") as f:
    f.write(sample_text)

# Load it
loader = TextLoader(sample_path, encoding="utf-8")
docs = loader.load()

print(f"Documents loaded: {len(docs)}")
print(f"Content preview:  {docs[0].page_content[:100]}...")
print(f"Metadata:         {docs[0].metadata}")
print(f"Content length:   {len(docs[0].page_content)} characters")

Documents loaded: 1
Content preview:  Introduction to Document Loaders

Document loaders are the entry point of any RAG pipeline. They ext...
Metadata:         {'source': '/var/folders/5p/1ncbfs612_z26t8m9d2j0rlr0000gn/T/sample_doc.txt'}
Content length:   498 characters


### Experiment 2B: Loading a CSV File

In [5]:
from langchain_community.document_loaders import CSVLoader
import csv

# Create a sample CSV
csv_path = os.path.join(tempfile.gettempdir(), "splitters.csv")
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["Splitter", "Best_For", "Speed"])
    writer.writerow(["RecursiveCharacter", "General text", "Very fast"])
    writer.writerow(["Token", "Token-critical apps", "Fast"])
    writer.writerow(["MarkdownHeader", "Structured docs", "Fast"])
    writer.writerow(["Semantic", "Transcripts, mixed text", "Slow"])

# Each CSV row becomes a separate Document
loader = CSVLoader(csv_path, encoding="utf-8")
docs = loader.load()

print(f"Documents loaded: {len(docs)} (one per row)\n")
for i, doc in enumerate(docs):
    print(f"Row {i+1}:")
    print(f"  Content:  {doc.page_content}")
    print(f"  Metadata: {doc.metadata}")
    print()

Documents loaded: 4 (one per row)

Row 1:
  Content:  Splitter: RecursiveCharacter
Best_For: General text
Speed: Very fast
  Metadata: {'source': '/var/folders/5p/1ncbfs612_z26t8m9d2j0rlr0000gn/T/splitters.csv', 'row': 0}

Row 2:
  Content:  Splitter: Token
Best_For: Token-critical apps
Speed: Fast
  Metadata: {'source': '/var/folders/5p/1ncbfs612_z26t8m9d2j0rlr0000gn/T/splitters.csv', 'row': 1}

Row 3:
  Content:  Splitter: MarkdownHeader
Best_For: Structured docs
Speed: Fast
  Metadata: {'source': '/var/folders/5p/1ncbfs612_z26t8m9d2j0rlr0000gn/T/splitters.csv', 'row': 2}

Row 4:
  Content:  Splitter: Semantic
Best_For: Transcripts, mixed text
Speed: Slow
  Metadata: {'source': '/var/folders/5p/1ncbfs612_z26t8m9d2j0rlr0000gn/T/splitters.csv', 'row': 3}



### Experiment 2C: Loading a JSON File

In [6]:
# Loading JSON without the jq dependency (jq is hard to install on Windows)
# Simple approach: read JSON manually and create Document objects

json_data = [
    {"title": "RecursiveCharacterTextSplitter", "type": "splitter",
     "description": "The default and most commonly used splitter. Uses hierarchical separators."},
    {"title": "CharacterTextSplitter", "type": "splitter",
     "description": "Simplest splitter — splits on a single separator only."},
    {"title": "TokenTextSplitter", "type": "splitter",
     "description": "Splits based on token count for precise control over LLM input sizes."},
]

# Create Document objects directly from JSON — no external loader needed
docs = [
    Document(
        page_content=item["description"],
        metadata={"title": item["title"], "type": item["type"], "source": "splitters.json"}
    )
    for item in json_data
]

print(f"Documents loaded: {len(docs)}\n")
for doc in docs:
    print(f"  [{doc.metadata.get('title')}]")
    print(f"  Content:  {doc.page_content}")
    print(f"  Metadata: {doc.metadata}")
    print()

Documents loaded: 3

  [RecursiveCharacterTextSplitter]
  Content:  The default and most commonly used splitter. Uses hierarchical separators.
  Metadata: {'title': 'RecursiveCharacterTextSplitter', 'type': 'splitter', 'source': 'splitters.json'}

  [CharacterTextSplitter]
  Content:  Simplest splitter — splits on a single separator only.
  Metadata: {'title': 'CharacterTextSplitter', 'type': 'splitter', 'source': 'splitters.json'}

  [TokenTextSplitter]
  Content:  Splits based on token count for precise control over LLM input sizes.
  Metadata: {'title': 'TokenTextSplitter', 'type': 'splitter', 'source': 'splitters.json'}



---

## 3. Web-Based Document Loaders

| Loader | Method | Best For |
|--------|--------|----------|
| `WebBaseLoader` | BeautifulSoup | Simple web pages, static HTML |
| `UnstructuredURLLoader` | Unstructured | Complex pages with better extraction |
| `SeleniumURLLoader` | Selenium | JavaScript-rendered pages (SPAs) |
| `PlaywrightURLLoader` | Playwright | Modern JS-heavy sites |

**Best practices:**
- Respect `robots.txt` and rate limits
- Add delays between requests to avoid IP blocking
- Cache downloaded pages during development

### Experiment 3: Loading a Web Page

In [7]:
from langchain_community.document_loaders import WebBaseLoader

# Load a simple web page (Python docs have clean HTML)
loader = WebBaseLoader("https://docs.python.org/3/faq/general.html")
docs = loader.load()

print(f"Documents loaded: {len(docs)}")
print(f"Content length:   {len(docs[0].page_content)} characters")
print(f"Metadata keys:    {list(docs[0].metadata.keys())}")
print(f"\nFirst 300 chars:")
print(docs[0].page_content[:3000].strip())

USER_AGENT environment variable not set, consider setting it to identify your requests.


Documents loaded: 1
Content length:   20269 characters
Metadata keys:    ['source', 'title', 'description', 'language']

First 300 chars:
General Python FAQ — Python 3.14.3 documentation




















































    Theme
    
Auto
Light
Dark



Table of Contents

General Python FAQ
General Information
Python in the real world





Previous topic
Python Frequently Asked Questions


Next topic
Programming FAQ



This page

Report a bug
Improve this page

Show source
        







Navigation


index

modules |

next |

previous |

Python »







3.14.3 Documentation »
    
Python Frequently Asked Questions »
General Python FAQ







                     |
                


    Theme
    
Auto
Light
Dark

 |







General Python FAQ¶

Contents

General Python FAQ

General Information

What is Python?
What is the Python Software Foundation?
Are there copyright restrictions on the use of Python?
Why was Python created in the first place?
What is Python good 

---

## 4. Directory Loader — Batch Loading Multiple Files

`DirectoryLoader` automatically selects the right loader based on file extension.

**Key options:**
- `glob` — Filter files by pattern (e.g., `"**/*.txt"`)
- `show_progress` — Display a progress bar
- `use_multithreading` — Parallel loading for speed
- `recursive` — Include subdirectories

### Experiment 4: Loading a Directory of Files

In [8]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

# Create a temp directory with multiple text files
doc_dir = os.path.join(tempfile.gettempdir(), "langchain_docs")
os.makedirs(doc_dir, exist_ok=True)

files = {
    "loaders.txt": "Document loaders extract text from PDFs, web pages, databases, and more into Document objects.",
    "splitters.txt": "Text splitters break large documents into smaller chunks for embeddings and retrieval.",
    "embeddings.txt": "Embeddings convert text chunks into numerical vectors for similarity search.",
    "retrievers.txt": "Retrievers find the most relevant chunks from a vector store given a user query.",
}

for name, content in files.items():
    with open(os.path.join(doc_dir, name), "w", encoding="utf-8") as f:
        f.write(content)

# Load all .txt files from the directory
loader = DirectoryLoader(
    doc_dir,
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    show_progress=True
)

docs = loader.load()

print(f"\nLoaded {len(docs)} documents from directory:\n")
for doc in docs:
    filename = os.path.basename(doc.metadata["source"])
    print(f"  {filename}: {doc.page_content[:70]}...")

100%|██████████| 4/4 [00:00<00:00, 984.93it/s]


Loaded 4 documents from directory:

  splitters.txt: Text splitters break large documents into smaller chunks for embedding...
  embeddings.txt: Embeddings convert text chunks into numerical vectors for similarity s...
  retrievers.txt: Retrievers find the most relevant chunks from a vector store given a u...
  loaders.txt: Document loaders extract text from PDFs, web pages, databases, and mor...


---

## 5. Custom Document Loaders

When built-in loaders don't fit, extend `BaseLoader` and implement `lazy_load()`.

**When to build custom loaders:**
- Proprietary system with a custom API
- Special preprocessing needed (cleaning, PII removal)
- Built-in loader doesn't extract the metadata you need
- Database schema specific to your application

### Experiment 5: Building a Custom Loader

In [9]:
from langchain_core.document_loaders import BaseLoader
from typing import Iterator


class FAQLoader(BaseLoader):
    """Custom loader that parses Q&A pairs from a simple text format."""

    def __init__(self, file_path: str, category: str = "general"):
        self.file_path = file_path
        self.category = category

    def lazy_load(self) -> Iterator[Document]:
        """Yield one Document per Q&A pair."""
        with open(self.file_path, "r", encoding="utf-8") as f:
            content = f.read()

        # Split on blank lines to get individual Q&A blocks
        blocks = content.strip().split("\n\n")

        for i, block in enumerate(blocks):
            lines = block.strip().split("\n")
            if len(lines) >= 2:
                question = lines[0].replace("Q: ", "")
                answer = " ".join(line.replace("A: ", "") for line in lines[1:])
                yield Document(
                    page_content=f"{question}\n{answer}",
                    metadata={
                        "source": self.file_path,
                        "category": self.category,
                        "faq_index": i,
                        "question": question
                    }
                )


# Create a sample FAQ file
faq_content = """Q: What is a Document Loader?
A: A component that reads data from a source and returns it as a list of Document objects.

Q: What is a Text Splitter?
A: A component that breaks text into smaller chunks for embeddings and retrieval.

Q: What is the best default splitter?
A: RecursiveCharacterTextSplitter with chunk_size=1000 and chunk_overlap=200."""

faq_path = os.path.join(tempfile.gettempdir(), "faq.txt")
with open(faq_path, "w", encoding="utf-8") as f:
    f.write(faq_content)

# Use the custom loader
loader = FAQLoader(faq_path, category="langchain")
docs = list(loader.lazy_load())

print(f"Loaded {len(docs)} FAQ entries:\n")
for doc in docs:
    print(f"  Q: {doc.metadata['question']}")
    print(f"  Content: {doc.page_content}")
    print(f"  Metadata: {doc.metadata}")
    print()

Loaded 3 FAQ entries:

  Q: What is a Document Loader?
  Content: What is a Document Loader?
A component that reads data from a source and returns it as a list of Document objects.
  Metadata: {'source': '/var/folders/5p/1ncbfs612_z26t8m9d2j0rlr0000gn/T/faq.txt', 'category': 'langchain', 'faq_index': 0, 'question': 'What is a Document Loader?'}

  Q: What is a Text Splitter?
  Content: What is a Text Splitter?
A component that breaks text into smaller chunks for embeddings and retrieval.
  Metadata: {'source': '/var/folders/5p/1ncbfs612_z26t8m9d2j0rlr0000gn/T/faq.txt', 'category': 'langchain', 'faq_index': 1, 'question': 'What is a Text Splitter?'}

  Q: What is the best default splitter?
  Content: What is the best default splitter?
RecursiveCharacterTextSplitter with chunk_size=1000 and chunk_overlap=200.
  Metadata: {'source': '/var/folders/5p/1ncbfs612_z26t8m9d2j0rlr0000gn/T/faq.txt', 'category': 'langchain', 'faq_index': 2, 'question': 'What is the best default splitter?'}



---

## 6. Text Splitters — Why Splitting Matters

Loaded documents are almost never in the right form for LLM processing:

- Embedding models have input size limits (typically 512-8192 tokens)
- LLM context windows are finite
- Retrieval precision improves with smaller, focused chunks
- Smaller chunks = fewer tokens per query = lower cost

### The Splitting Dilemma

| Problem | Chunks Too Small | Chunks Too Large |
|---------|-----------------|------------------|
| Context | Lose context — fragments are meaningless alone | Embedding becomes too general |
| Retrieval | Snippets can't answer questions | Low precision — matches many queries, relevant to none |
| Embedding | Not enough text for a meaningful vector | Wastes context window space |

**Sweet spot:** 500-1000 characters (200-500 tokens) for most applications.

### Key Parameters (All Splitters)

| Parameter | Description | Typical Values |
|-----------|-------------|----------------|
| `chunk_size` | Maximum size of each chunk | 500-2000 characters |
| `chunk_overlap` | Overlapping chars between chunks | 50-200 (10-20% of chunk_size) |
| `length_function` | How to measure chunk size | `len` or custom token counter |
| `separators` | Ordered list of split boundaries | Varies by splitter |

### Why Overlap Matters

```
Without overlap:
  Chunk 1: "...Python supports multiple data types including"
  Chunk 2: "integers, floats, strings, and booleans. Each type..."
  --> Complete sentence split across two chunks!

With overlap (200 chars):
  Chunk 1: "...Python supports multiple data types including integers, floats,"
  Chunk 2: "data types including integers, floats, strings, and booleans..."
  --> Boundary content appears in both chunks
```

---

## 7. RecursiveCharacterTextSplitter — The Default Choice

Attempts to split using a **hierarchy of separators**, trying the largest meaningful boundary first:

1. `"\n\n"` — Paragraph break (tries this first)
2. `"\n"` — Line break
3. `" "` — Space (word boundary)
4. `""` — Character-by-character (last resort)

**Algorithm:** Try paragraph breaks -> if chunks still too large, try line breaks -> if still too large, try spaces -> merge small chunks -> apply overlap.

### Experiment 7A: Basic Recursive Splitting

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# A multi-paragraph document
long_text = """Document Loaders in LangChain

Document loaders are components that read data from a source and return it as a list of Document objects. Every loader produces the same standardized output regardless of the source format.

The Document object has two fields: page_content (the extracted text) and metadata (information about the source like filename, page number, URL, and timestamp).

Text Splitters in LangChain

Text splitters break extracted text into optimally-sized chunks. The chunks should be small enough for embedding models but large enough to retain meaningful context.

The most important parameters are chunk_size (maximum characters per chunk) and chunk_overlap (characters shared between consecutive chunks to preserve boundary context).

Choosing the Right Splitter

RecursiveCharacterTextSplitter is the recommended default for general text. It uses a hierarchy of separators to keep paragraphs, sentences, and words intact.

For code, use language-specific splitters. For markdown, use MarkdownHeaderTextSplitter. For highest quality, use SemanticChunker with embeddings."""

# Split with RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=30,
    length_function=len,
)

chunks = splitter.split_text(long_text)

print(f"Original: {len(long_text)} chars")
print(f"Chunks:   {len(chunks)}")
print(f"Config:   chunk_size=200, overlap=30\n")

for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ({len(chunk)} chars) ---")
    print(chunk)
    print()

Original: 1089 chars
Chunks:   7
Config:   chunk_size=200, overlap=30

--- Chunk 1 (29 chars) ---
Document Loaders in LangChain

--- Chunk 2 (189 chars) ---
Document loaders are components that read data from a source and return it as a list of Document objects. Every loader produces the same standardized output regardless of the source format.

--- Chunk 3 (190 chars) ---
The Document object has two fields: page_content (the extracted text) and metadata (information about the source like filename, page number, URL, and timestamp).

Text Splitters in LangChain

--- Chunk 4 (195 chars) ---
Text Splitters in LangChain

Text splitters break extracted text into optimally-sized chunks. The chunks should be small enough for embedding models but large enough to retain meaningful context.

--- Chunk 5 (170 chars) ---
The most important parameters are chunk_size (maximum characters per chunk) and chunk_overlap (characters shared between consecutive chunks to preserve boundary context).

--- Chu

### Experiment 7B: Splitting Documents (Preserving Metadata)

In [11]:
# When splitting Document objects, metadata is automatically preserved

original_doc = Document(
    page_content=long_text,
    metadata={"source": "langchain_guide.pdf", "page": 1, "author": "Tutorial"}
)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

# split_documents preserves and copies metadata to every chunk
doc_chunks = splitter.split_documents([original_doc])

print(f"Original: 1 document, {len(original_doc.page_content)} chars")
print(f"Split:    {len(doc_chunks)} chunks\n")

for i, chunk in enumerate(doc_chunks):
    print(f"Chunk {i+1}: {len(chunk.page_content)} chars")
    print(f"  Metadata: {chunk.metadata}")
    print(f"  Preview:  {chunk.page_content[:80]}...")
    print()

Original: 1 document, 1089 chars
Split:    6 chunks

Chunk 1: 220 chars
  Metadata: {'source': 'langchain_guide.pdf', 'page': 1, 'author': 'Tutorial'}
  Preview:  Document Loaders in LangChain

Document loaders are components that read data fr...

Chunk 2: 190 chars
  Metadata: {'source': 'langchain_guide.pdf', 'page': 1, 'author': 'Tutorial'}
  Preview:  The Document object has two fields: page_content (the extracted text) and metada...

Chunk 3: 195 chars
  Metadata: {'source': 'langchain_guide.pdf', 'page': 1, 'author': 'Tutorial'}
  Preview:  Text Splitters in LangChain

Text splitters break extracted text into optimally-...

Chunk 4: 199 chars
  Metadata: {'source': 'langchain_guide.pdf', 'page': 1, 'author': 'Tutorial'}
  Preview:  The most important parameters are chunk_size (maximum characters per chunk) and ...

Chunk 5: 187 chars
  Metadata: {'source': 'langchain_guide.pdf', 'page': 1, 'author': 'Tutorial'}
  Preview:  Choosing the Right Splitter

RecursiveCharacterTextSplitt

### Experiment 7C: Effect of Chunk Size on Splitting

In [12]:
# Compare different chunk sizes on the same text

sizes = [100, 200, 500, 1000]

print(f"Original text: {len(long_text)} characters\n")
print(f"{'Chunk Size':>12} | {'Overlap':>8} | {'Chunks':>6} | {'Avg Size':>8} | {'Min':>5} | {'Max':>5}")
print("-" * 65)

for size in sizes:
    overlap = int(size * 0.15)  # 15% overlap
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size,
        chunk_overlap=overlap
    )
    chunks = splitter.split_text(long_text)
    lengths = [len(c) for c in chunks]

    print(f"{size:>12} | {overlap:>8} | {len(chunks):>6} | "
          f"{sum(lengths)/len(lengths):>8.0f} | {min(lengths):>5} | {max(lengths):>5}")

Original text: 1089 characters

  Chunk Size |  Overlap | Chunks | Avg Size |   Min |   Max
-----------------------------------------------------------------
         100 |       15 |     16 |       72 |    21 |    98
         200 |       30 |      7 |      158 |    29 |   195
         500 |       75 |      3 |      381 |   335 |   412
        1000 |      150 |      2 |      544 |   146 |   941


---

## 8. CharacterTextSplitter — Single Separator

Splits on a **single separator** only (default: `"\n\n"`). Unlike `RecursiveCharacter`, it has **no fallback** — if a section exceeds `chunk_size`, it produces an oversized chunk.

**When to use:**
- Text with a reliable, consistent separator
- When you want exact control over boundaries
- Pre-formatted text where structure is guaranteed

### Experiment 8: CharacterTextSplitter vs Recursive

In [13]:
from langchain_text_splitters import CharacterTextSplitter

# CharacterTextSplitter — splits on paragraph breaks only
char_splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=200,
    chunk_overlap=0
)

# Recursive splitter for comparison
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=0
)

char_chunks = char_splitter.split_text(long_text)
recursive_chunks = recursive_splitter.split_text(long_text)

print("CharacterTextSplitter (single separator: paragraph breaks)")
print(f"  Chunks: {len(char_chunks)}")
for i, c in enumerate(char_chunks):
    marker = " [OVERSIZED!]" if len(c) > 200 else ""
    print(f"  Chunk {i+1}: {len(c)} chars{marker}")

print(f"\nRecursiveCharacterTextSplitter (hierarchical separators)")
print(f"  Chunks: {len(recursive_chunks)}")
for i, c in enumerate(recursive_chunks):
    marker = " [OVERSIZED!]" if len(c) > 200 else ""
    print(f"  Chunk {i+1}: {len(c)} chars{marker}")

print("\nKey difference: CharacterTextSplitter can produce oversized chunks!")
print("RecursiveCharacter falls back to smaller separators to enforce size.")

CharacterTextSplitter (single separator: paragraph breaks)
  Chunks: 7
  Chunk 1: 29 chars
  Chunk 2: 189 chars
  Chunk 3: 190 chars
  Chunk 4: 166 chars
  Chunk 5: 199 chars
  Chunk 6: 158 chars
  Chunk 7: 146 chars

RecursiveCharacterTextSplitter (hierarchical separators)
  Chunks: 7
  Chunk 1: 29 chars
  Chunk 2: 189 chars
  Chunk 3: 190 chars
  Chunk 4: 166 chars
  Chunk 5: 170 chars
  Chunk 6: 187 chars
  Chunk 7: 146 chars

Key difference: CharacterTextSplitter can produce oversized chunks!
RecursiveCharacter falls back to smaller separators to enforce size.


---

## 9. TokenTextSplitter — Token-Based Splitting

Characters and tokens don't have a fixed relationship:

| Text | Characters | Tokens |
|------|-----------|--------|
| `"Hello"` | 5 | 1 |
| `"Anthropomorphization"` | 20 | 4 |
| Code with special chars | Varies wildly | Unpredictable |

Character-based splitting with `chunk_size=1000` might produce chunks ranging from 200 to 800 tokens. Token-based splitting gives **precise control**.

### Experiment 9: Token-Based vs Character-Based

In [14]:
from langchain_text_splitters import TokenTextSplitter

# Token-based splitter (uses tiktoken by default)
token_splitter = TokenTextSplitter(
    chunk_size=50,       # 50 tokens per chunk
    chunk_overlap=10     # 10 tokens overlap
)

token_chunks = token_splitter.split_text(long_text)

print(f"Token-based splitting (chunk_size=50 tokens)")
print(f"Total chunks: {len(token_chunks)}\n")

for i, chunk in enumerate(token_chunks):
    # Approximate token count (tiktoken)
    words = len(chunk.split())
    print(f"Chunk {i+1}: {len(chunk)} chars, ~{words} words")
    print(f"  {chunk[:80]}...")
    print()

Token-based splitting (chunk_size=50 tokens)
Total chunks: 6

Chunk 1: 256 chars, ~41 words
  Document Loaders in LangChain

Document loaders are components that read data fr...

Chunk 2: 216 chars, ~30 words
   format.

The Document object has two fields: page_content (the extracted text) ...

Chunk 3: 246 chars, ~34 words
   Splitters in LangChain

Text splitters break extracted text into optimally-size...

Chunk 4: 250 chars, ~29 words
  The most important parameters are chunk_size (maximum characters per chunk) and ...

Chunk 5: 226 chars, ~30 words
  
RecursiveCharacterTextSplitter is the recommended default for general text. It ...

Chunk 6: 123 chars, ~13 words
  specific splitters. For markdown, use MarkdownHeaderTextSplitter. For highest qu...



---

## 10. Language-Specific Code Splitters

`RecursiveCharacterTextSplitter.from_language()` uses **language-specific boundaries** instead of generic text separators.

| Language | Method | Key Boundaries |
|----------|--------|----------------|
| Python | `Language.PYTHON` | `class`, `def`, `if`, `for` |
| JavaScript | `Language.JS` | `function`, `class`, `const` |
| Java | `Language.JAVA` | `class`, `public`, `private` |
| Markdown | `Language.MARKDOWN` | Headers (`#`, `##`, `###`) |

**Why generic splitters fail on code:** A generic splitter might break in the middle of a function, separating the signature from its body.

### Experiment 10: Python Code Splitting

In [15]:
from langchain_text_splitters import Language

python_code = '''
class DocumentProcessor:
    """Processes documents through loading and splitting."""

    def __init__(self, chunk_size=1000):
        self.chunk_size = chunk_size
        self.documents = []

    def load_file(self, file_path):
        """Load a single file into a Document."""
        with open(file_path, "r") as f:
            content = f.read()
        self.documents.append({
            "content": content,
            "source": file_path
        })
        return self

    def split_documents(self):
        """Split all loaded documents into chunks."""
        chunks = []
        for doc in self.documents:
            text = doc["content"]
            for i in range(0, len(text), self.chunk_size):
                chunks.append(text[i:i + self.chunk_size])
        return chunks


def create_pipeline(chunk_size=1000, overlap=200):
    """Create a document processing pipeline."""
    processor = DocumentProcessor(chunk_size)
    return processor


def calculate_statistics(chunks):
    """Calculate chunk statistics."""
    sizes = [len(c) for c in chunks]
    return {
        "count": len(chunks),
        "avg_size": sum(sizes) / len(sizes),
        "min_size": min(sizes),
        "max_size": max(sizes),
    }
'''

# Python-aware splitter
python_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON,
    chunk_size=300,
    chunk_overlap=30
)

# Generic splitter for comparison
generic_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=30
)

python_chunks = python_splitter.split_text(python_code)
generic_chunks = generic_splitter.split_text(python_code)

print("PYTHON-AWARE SPLITTER")
print("=" * 50)
for i, chunk in enumerate(python_chunks):
    # Check if chunk starts with a class/function definition
    first_line = chunk.strip().split("\n")[0]
    print(f"Chunk {i+1} ({len(chunk)} chars): {first_line}")

print(f"\nGENERIC SPLITTER")
print("=" * 50)
for i, chunk in enumerate(generic_chunks):
    first_line = chunk.strip().split("\n")[0]
    print(f"Chunk {i+1} ({len(chunk)} chars): {first_line}")

print("\nPython splitter keeps class/function definitions intact!")

PYTHON-AWARE SPLITTER
Chunk 1 (192 chars): class DocumentProcessor:
Chunk 2 (279 chars): def load_file(self, file_path):
Chunk 3 (287 chars): def split_documents(self):
Chunk 4 (13 chars): return chunks
Chunk 5 (166 chars): def create_pipeline(chunk_size=1000, overlap=200):
Chunk 6 (266 chars): def calculate_statistics(chunks):

GENERIC SPLITTER
Chunk 1 (192 chars): class DocumentProcessor:
Chunk 2 (279 chars): def load_file(self, file_path):
Chunk 3 (287 chars): def split_documents(self):
Chunk 4 (13 chars): return chunks
Chunk 5 (166 chars): def create_pipeline(chunk_size=1000, overlap=200):
Chunk 6 (266 chars): def calculate_statistics(chunks):

Python splitter keeps class/function definitions intact!


---

## 11. MarkdownHeaderTextSplitter — Structure-Aware Splitting

Splits at **Markdown headers** and attaches the header hierarchy as metadata.

```
Input:  # Intro           Output:  Doc 1: "Some text"
        Some text                  metadata: {"H1": "Intro"}
        ## Methods                 Doc 2: "Details here"
        Details here               metadata: {"H1": "Intro", "H2": "Methods"}
```

**Why this is powerful:**
- Semantic coherence — each chunk is from a single logical section
- Rich metadata — full header hierarchy shows where content lives
- Natural boundaries — headers are the author's own topic indicators

### Experiment 11: Markdown Header-Based Splitting

In [16]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

markdown_doc = """# Document Processing Pipeline

This guide covers the complete document processing pipeline for RAG applications.

## Document Loaders

Document loaders extract text from diverse sources into standardized Document objects. LangChain provides over 100 loaders for files, web pages, databases, and cloud storage.

### PDF Loaders

PDFs are the most common enterprise format. Choose PyPDFLoader for simple text, PDFPlumberLoader for tables, and UnstructuredPDFLoader for complex layouts.

### Web Loaders

WebBaseLoader handles static HTML. For JavaScript-rendered content, use SeleniumURLLoader or PlaywrightURLLoader.

## Text Splitters

Text splitters break loaded documents into chunks optimized for embeddings and retrieval.

### RecursiveCharacterTextSplitter

The recommended default. Uses a hierarchy of separators to keep paragraphs, sentences, and words together.

### SemanticChunker

Uses embedding similarity to split at natural topic boundaries. Highest quality but slowest.

## Best Practices

Start with RecursiveCharacterTextSplitter, chunk_size=1000, chunk_overlap=200. Test with your actual data."""

# Define headers to split on
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on
)

md_chunks = md_splitter.split_text(markdown_doc)

print(f"Markdown document split into {len(md_chunks)} sections:\n")
for i, chunk in enumerate(md_chunks):
    print(f"--- Section {i+1} ---")
    print(f"  Headers:  {chunk.metadata}")
    print(f"  Content:  {chunk.page_content[:100]}...")
    print()

Markdown document split into 8 sections:

--- Section 1 ---
  Headers:  {'Header 1': 'Document Processing Pipeline'}
  Content:  This guide covers the complete document processing pipeline for RAG applications....

--- Section 2 ---
  Headers:  {'Header 1': 'Document Processing Pipeline', 'Header 2': 'Document Loaders'}
  Content:  Document loaders extract text from diverse sources into standardized Document objects. LangChain pro...

--- Section 3 ---
  Headers:  {'Header 1': 'Document Processing Pipeline', 'Header 2': 'Document Loaders', 'Header 3': 'PDF Loaders'}
  Content:  PDFs are the most common enterprise format. Choose PyPDFLoader for simple text, PDFPlumberLoader for...

--- Section 4 ---
  Headers:  {'Header 1': 'Document Processing Pipeline', 'Header 2': 'Document Loaders', 'Header 3': 'Web Loaders'}
  Content:  WebBaseLoader handles static HTML. For JavaScript-rendered content, use SeleniumURLLoader or Playwri...

--- Section 5 ---
  Headers:  {'Header 1': 'Document Proces

---

## 12. Two-Pass Splitting — Best of Both Worlds

The most effective approach for structured documents:

```
Raw Document
    |
Pass 1: MarkdownHeaderTextSplitter
    --> Section-level chunks with header metadata
    |
Pass 2: RecursiveCharacterTextSplitter (oversized chunks only)
    --> All chunks within target size, metadata preserved
    |
Embedding Pipeline
```

### Experiment 12: Two-Pass Splitting Strategy

In [17]:
# Pass 1: Structure-aware splitting by headers
md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("#", "Header 1"),
        ("##", "Header 2"),
        ("###", "Header 3"),
    ]
)

# Pass 2: Size enforcement for oversized sections
size_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=20
)

# Step 1: Split by headers
header_chunks = md_splitter.split_text(markdown_doc)

print("PASS 1 — Header-based splitting")
print(f"Sections: {len(header_chunks)}")
for i, chunk in enumerate(header_chunks):
    size_status = "OVERSIZED" if len(chunk.page_content) > 150 else "OK"
    print(f"  Section {i+1}: {len(chunk.page_content)} chars [{size_status}] — {chunk.metadata}")

# Step 2: Split oversized sections, keep small ones intact
final_chunks = []
for chunk in header_chunks:
    if len(chunk.page_content) > 150:
        # Split this oversized section further
        sub_chunks = size_splitter.split_documents([chunk])
        final_chunks.extend(sub_chunks)
    else:
        final_chunks.append(chunk)

print(f"\nPASS 2 — Size enforcement")
print(f"Final chunks: {len(final_chunks)}")
for i, chunk in enumerate(final_chunks):
    print(f"  Chunk {i+1}: {len(chunk.page_content)} chars — {chunk.metadata}")
    print(f"    {chunk.page_content[:80]}...")

PASS 1 — Header-based splitting
Sections: 8
  Section 1: 81 chars [OK] — {'Header 1': 'Document Processing Pipeline'}
  Section 2: 174 chars [OVERSIZED] — {'Header 1': 'Document Processing Pipeline', 'Header 2': 'Document Loaders'}
  Section 3: 155 chars [OVERSIZED] — {'Header 1': 'Document Processing Pipeline', 'Header 2': 'Document Loaders', 'Header 3': 'PDF Loaders'}
  Section 4: 113 chars [OK] — {'Header 1': 'Document Processing Pipeline', 'Header 2': 'Document Loaders', 'Header 3': 'Web Loaders'}
  Section 5: 89 chars [OK] — {'Header 1': 'Document Processing Pipeline', 'Header 2': 'Text Splitters'}
  Section 6: 106 chars [OK] — {'Header 1': 'Document Processing Pipeline', 'Header 2': 'Text Splitters', 'Header 3': 'RecursiveCharacterTextSplitter'}
  Section 7: 92 chars [OK] — {'Header 1': 'Document Processing Pipeline', 'Header 2': 'Text Splitters', 'Header 3': 'SemanticChunker'}
  Section 8: 106 chars [OK] — {'Header 1': 'Document Processing Pipeline', 'Header 2': 'Best Practices'

---

## 13. Metadata Enrichment During Splitting

Enrich chunk metadata beyond what the loader provides:

| Metadata | Purpose |
|----------|----------|
| `chunk_index` | Position within the document (for re-ordering) |
| `total_chunks` | Total chunks from this document |
| `doc_type` | Classification (policy, guide, faq) |
| `last_updated` | When the source was last modified |

### Experiment 13: Adding Chunk Metadata

In [18]:
from datetime import datetime


def enrich_chunks(documents, doc_type="general"):
    """Add chunk-specific metadata to a list of Document chunks."""
    total = len(documents)
    enriched = []

    for i, doc in enumerate(documents):
        doc.metadata.update({
            "chunk_index": i,
            "total_chunks": total,
            "doc_type": doc_type,
            "char_count": len(doc.page_content),
            "word_count": len(doc.page_content.split()),
            "indexed_at": datetime.now().isoformat(),
        })
        enriched.append(doc)

    return enriched


# Split a document
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
raw_chunks = splitter.split_documents([
    Document(
        page_content=long_text,
        metadata={"source": "langchain_guide.pdf", "page": 1}
    )
])

# Enrich with additional metadata
enriched_chunks = enrich_chunks(raw_chunks, doc_type="tutorial")

print(f"Enriched {len(enriched_chunks)} chunks:\n")
for chunk in enriched_chunks[:3]:
    print(f"Chunk {chunk.metadata['chunk_index'] + 1}/{chunk.metadata['total_chunks']}:")
    for key, val in chunk.metadata.items():
        print(f"  {key}: {val}")
    print(f"  preview: {chunk.page_content[:60]}...")
    print()

Enriched 7 chunks:

Chunk 1/7:
  source: langchain_guide.pdf
  page: 1
  chunk_index: 0
  total_chunks: 7
  doc_type: tutorial
  char_count: 29
  word_count: 4
  indexed_at: 2026-03-08T18:19:45.761362
  preview: Document Loaders in LangChain...

Chunk 2/7:
  source: langchain_guide.pdf
  page: 1
  chunk_index: 1
  total_chunks: 7
  doc_type: tutorial
  char_count: 189
  word_count: 31
  indexed_at: 2026-03-08T18:19:45.761416
  preview: Document loaders are components that read data from a source...

Chunk 3/7:
  source: langchain_guide.pdf
  page: 1
  chunk_index: 2
  total_chunks: 7
  doc_type: tutorial
  char_count: 190
  word_count: 27
  indexed_at: 2026-03-08T18:19:45.761420
  preview: The Document object has two fields: page_content (the extrac...



---

## 14. Chunk Size Optimization with LLM Evaluation

### Guidelines by Use Case

| Use Case | Chunk Size | Overlap | Rationale |
|----------|-----------|---------|----------|
| General Q&A | 500-1000 chars | 100-200 | Balance context and precision |
| Detailed analysis | 1000-2000 chars | 200-400 | More context for nuanced answers |
| Code search | 1000-1500 chars | 100-200 | Keep complete functions together |
| Legal/medical | 500-800 chars | 150-200 | Precision critical |
| Conversational | 200-500 chars | 50-100 | Short exchanges |
| Summarization | 2000-4000 chars | 200-400 | Larger context for quality |

**Default recommendation:** Start with `chunk_size=1000`, `chunk_overlap=200`.

### Experiment 14: Comparing Chunk Sizes for RAG Quality

In [19]:
# Simulate how different chunk sizes affect retrieval quality
# We'll split a knowledge base and ask the LLM to answer using each chunk size

knowledge = """LangChain Document Loaders

Document loaders are LangChain components that read data from a source and return it as a list of Document objects. Every loader produces the same standardized output with page_content and metadata fields.

PDF loaders are the most common. PyPDFLoader is fast and lightweight, good for simple text PDFs. PDFPlumberLoader excels at table extraction. UnstructuredPDFLoader handles complex layouts with mixed content.

Web loaders include WebBaseLoader for static HTML, SeleniumURLLoader for JavaScript-rendered pages, and RecursiveUrlLoader for crawling entire sites. Always respect robots.txt and add delays between requests.

DirectoryLoader batch-loads all files from a folder, auto-selecting loaders by extension. Use glob patterns to filter, show_progress for visibility, and use_multithreading for speed.

Custom loaders extend BaseLoader and implement lazy_load() — a generator that yields Document objects one at a time. This pattern is essential for large datasets to avoid loading everything into memory.

Text splitters break loaded documents into chunks. RecursiveCharacterTextSplitter is the best default, using hierarchical separators. TokenTextSplitter gives precise token control. MarkdownHeaderTextSplitter preserves document structure."""

question = "How do I load PDF files with tables in LangChain?"

# Try different chunk sizes
configs = [
    {"name": "Small (100)",  "size": 100,  "overlap": 15},
    {"name": "Medium (300)", "size": 300,  "overlap": 45},
    {"name": "Large (600)",  "size": 600,  "overlap": 90},
    {"name": "XL (1000)",    "size": 1000, "overlap": 150},
]

answer_chain = (
    ChatPromptTemplate.from_messages([
        ("system",
         "Answer the question using ONLY the provided context. "
         "Be specific. 2 sentences max.\n\nContext: {context}"),
        ("human", "{question}")
    ]) | llm | StrOutputParser()
)

print(f"Question: {question}\n")

for config in configs:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=config["size"],
        chunk_overlap=config["overlap"]
    )
    chunks = splitter.split_text(knowledge)

    # Simulate retrieval: find the most relevant chunk (simple keyword match)
    scored = [(c, sum(1 for w in ["pdf", "table", "loader"] if w in c.lower())) for c in chunks]
    best_chunk = max(scored, key=lambda x: x[1])[0]

    answer = answer_chain.invoke({"context": best_chunk, "question": question})

    print(f"{config['name']:>15} | {len(chunks)} chunks | best chunk: {len(best_chunk)} chars")
    print(f"{'':>15}   Answer: {answer}")
    print()

Question: How do I load PDF files with tables in LangChain?

    Small (100) | 18 chunks | best chunk: 93 chars
                  Answer: To load PDF files with tables, you can use the `UnstructuredPDFLoader` from LangChain, which excels at handling complex layouts and extracting structured data.

   Medium (300) | 6 chunks | best chunk: 207 chars
                  Answer: To load PDF files with tables in LangChain, you can use the PDFPlumberLoader.

    Large (600) | 3 chunks | best chunk: 442 chars
                  Answer: You can use the PDFPlumberLoader in LangChain to load PDF files with tables. This loader excels at table extraction, allowing you to easily retrieve the table data from the PDF file.

      XL (1000) | 2 chunks | best chunk: 836 chars
                  Answer: You can use the PDFPlumberLoader to load PDF files with tables in LangChain. It excels at table extraction, allowing you to retrieve the table data and convert it into a structured format.



---

## 15. Complete RAG Pipeline — Loaders + Splitters + LLM

Putting it all together: load documents, split them, simulate retrieval, and answer questions.

### Experiment 15: End-to-End Document Q&A Pipeline

In [20]:
# ============================================================
#  END-TO-END: Load -> Split -> Retrieve -> Answer
# ============================================================

# Step 1: Load documents (from our temp directory)
loader = DirectoryLoader(
    doc_dir,
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)
raw_docs = loader.load()
print(f"Step 1 — Loaded {len(raw_docs)} documents")

# Step 2: Split into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=30
)
chunks = splitter.split_documents(raw_docs)

# Enrich metadata
chunks = enrich_chunks(chunks, doc_type="rag_knowledge_base")
print(f"Step 2 — Split into {len(chunks)} chunks")

# Step 3: Simple keyword retrieval (in production, use vector store)
def simple_retrieve(query, chunks, top_k=2):
    """Simple keyword-based retrieval for demonstration."""
    query_words = set(query.lower().split())
    scored = []
    for chunk in chunks:
        chunk_words = set(chunk.page_content.lower().split())
        score = len(query_words & chunk_words)
        scored.append((chunk, score))
    scored.sort(key=lambda x: x[1], reverse=True)
    return [c for c, s in scored[:top_k]]

# Step 4: Answer questions using retrieved context
qa_chain = (
    ChatPromptTemplate.from_messages([
        ("system",
         "Answer the question using ONLY the provided context. "
         "Cite the source file. Be concise.\n\n"
         "Context:\n{context}"),
        ("human", "{question}")
    ]) | llm | StrOutputParser()
)

questions = [
    "What do document loaders do?",
    "How do text splitters work?",
    "What are embeddings used for?",
]

print(f"\n{'=' * 60}")
print("Step 3+4 — Retrieve & Answer")
print(f"{'=' * 60}")

for q in questions:
    retrieved = simple_retrieve(q, chunks, top_k=2)
    context = "\n\n".join(
        f"[Source: {os.path.basename(c.metadata['source'])}] {c.page_content}"
        for c in retrieved
    )

    answer = qa_chain.invoke({"context": context, "question": q})

    print(f"\nQ: {q}")
    print(f"Retrieved from: {[os.path.basename(c.metadata['source']) for c in retrieved]}")
    print(f"A: {answer}")

Step 1 — Loaded 4 documents
Step 2 — Split into 4 chunks

Step 3+4 — Retrieve & Answer

Q: What do document loaders do?
Retrieved from: ['loaders.txt', 'splitters.txt']
A: According to the context, document loaders extract text from PDFs, web pages, databases, and more into Document objects. [Source: loaders.txt]

Q: How do text splitters work?
Retrieved from: ['splitters.txt', 'embeddings.txt']
A: Source: [Source: splitters.txt] 

Text splitters break large documents into smaller chunks for embeddings and retrieval.

Q: What are embeddings used for?
Retrieved from: ['splitters.txt', 'embeddings.txt']
A: According to the context, embeddings are used for "similarity search". 

Source: embeddings.txt


---

## Splitter Comparison — Complete Reference

| Splitter | Split Logic | Semantic | Speed | Size Control | Best For |
|----------|------------|----------|-------|-------------|----------|
| **RecursiveChar** | Hierarchical separators | Low | Very fast | Precise | General — best default |
| **Character** | Single separator | None | Very fast | Limited | Simple, uniform text |
| **Token** | Token count | None | Fast | Exact | Token-critical apps |
| **Language** | Syntax boundaries | Medium | Fast | Good | Source code |
| **MarkdownHeader** | Header hierarchy | High | Fast | Variable | Markdown docs |
| **HTMLHeader** | HTML header tags | High | Fast | Variable | Web content |
| **Semantic** | Embedding similarity | Highest | Slow | None | Transcripts, mixed text |

### Decision Framework

1. **What is the content type?**
   - Source code -> Language-specific splitter
   - Markdown -> MarkdownHeaderTextSplitter + RecursiveCharacter
   - HTML -> HTMLHeaderTextSplitter + RecursiveCharacter
   - General text -> Step 2

2. **How important is semantic precision?**
   - Critical -> SemanticChunker (with size-based second pass)
   - Standard -> RecursiveCharacterTextSplitter
   - Not important -> CharacterTextSplitter

3. **Need exact token control?**
   - Yes -> TokenTextSplitter
   - No -> Character-based is fine

### Best Practices

| Practice | Description |
|----------|-------------|
| Always inspect loaded documents | Print first few docs to verify quality |
| Handle loading errors gracefully | Use try/except, log failures |
| Preserve metadata through splitting | Ensure loader metadata passes to all child chunks |
| Add chunk-specific metadata | Include chunk_index and total_chunks |
| Start with RecursiveCharacter | Optimize to specialized splitters only if needed |
| Use two-pass for structured docs | Header-based first, then size-based |
| Test chunk quality manually | Read 20-30 random chunks — do they make sense standalone? |
| Benchmark chunk sizes empirically | Test 3-4 sizes with your actual queries |

### Common Pitfalls

| Pitfall | Problem | Solution |
|---------|---------|----------|
| Wrong PDF loader | Tables as gibberish | PDFPlumber for tables |
| No overlap | Info lost at boundaries | Set overlap to 10-20% of chunk_size |
| Chunks too small | Snippets lack context | Increase chunk_size (min 300-500) |
| Chunks too large | Low retrieval precision | Decrease chunk_size (max 1500-2000) |
| Ignoring metadata | Can't trace answers to sources | Preserve and enrich metadata |
| Generic splitter for code | Functions split mid-definition | Use Language-specific splitters |

---

## 16. Sandbox — Try It Yourself!

In [21]:
# ============================================================
#  SANDBOX — Experiment with loaders and splitters!
# ============================================================

# Try changing these parameters and see how results change:

sandbox_text = """Artificial Intelligence in Healthcare

AI is transforming healthcare through early disease detection, drug discovery, and personalized treatment plans. Machine learning models can analyze medical images with accuracy matching or exceeding human specialists.

Natural Language Processing enables extraction of insights from clinical notes, research papers, and patient records. This helps identify patterns that would be impossible for humans to detect across millions of documents.

Challenges remain in data privacy, model interpretability, and regulatory compliance. Healthcare AI systems must be explainable and auditable to gain trust from clinicians and patients."""

# Experiment with different settings
CHUNK_SIZE = 150     # Try: 100, 200, 300, 500
CHUNK_OVERLAP = 20   # Try: 0, 20, 50, 100

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

chunks = splitter.split_text(sandbox_text)

print(f"Settings: chunk_size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP}")
print(f"Result:   {len(chunks)} chunks\n")

for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ({len(chunk)} chars) ---")
    print(chunk)
    print()

# Ask the LLM about a specific chunk
if chunks:
    qa = (
        ChatPromptTemplate.from_messages([
            ("system", "Answer based on this context only. 1-2 sentences.\n\nContext: {context}"),
            ("human", "{question}")
        ]) | llm | StrOutputParser()
    )
    answer = qa.invoke({
        "context": chunks[0],
        "question": "What is this about?"
    })
    print(f"LLM on Chunk 1: {answer}")

Settings: chunk_size=150, overlap=20
Result:   7 chunks

--- Chunk 1 (37 chars) ---
Artificial Intelligence in Healthcare

--- Chunk 2 (148 chars) ---
AI is transforming healthcare through early disease detection, drug discovery, and personalized treatment plans. Machine learning models can analyze

--- Chunk 3 (88 chars) ---
models can analyze medical images with accuracy matching or exceeding human specialists.

--- Chunk 4 (146 chars) ---
Natural Language Processing enables extraction of insights from clinical notes, research papers, and patient records. This helps identify patterns

--- Chunk 5 (93 chars) ---
identify patterns that would be impossible for humans to detect across millions of documents.

--- Chunk 6 (149 chars) ---
Challenges remain in data privacy, model interpretability, and regulatory compliance. Healthcare AI systems must be explainable and auditable to gain

--- Chunk 7 (53 chars) ---
auditable to gain trust from clinicians and patients.

LLM on Chunk 1: Artific

---

## Key Takeaways

| Concept | What to Remember |
|---------|------------------|
| **Document Object** | Universal format: `page_content` + `metadata` |
| **Metadata** | First-class concern — enables attribution, filtering, access control |
| **PDF Loaders** | PyPDFLoader (simple), PDFPlumber (tables), Unstructured (complex) |
| **DirectoryLoader** | Batch loading with auto-selected loaders by extension |
| **Custom Loaders** | Extend `BaseLoader`, implement `lazy_load()` |
| **RecursiveCharacter** | Best default — hierarchical separators, start here |
| **CharacterText** | Single separator — no fallback for oversized chunks |
| **TokenText** | Precise token control for embedding model limits |
| **Language Splitters** | Code-aware — keeps functions/classes intact |
| **MarkdownHeader** | Structure-aware — header hierarchy as metadata |
| **Two-Pass Splitting** | Structure-first + size enforcement = best results |
| **Chunk Size** | Start with 1000 chars / 200 overlap, test empirically |
| **Metadata Enrichment** | Add chunk_index, total_chunks, doc_type during splitting |